In [1]:
pip install numpy pandas scikit-learn scipy shap lime python-sat matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 65.8 MB/s  0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=ec3e165c6031c0a12c5eeb732f2c5554644635b562679cf863d55c29e5de40cc
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [lime]


In [4]:
#!/usr/bin/env python3
"""
run_all.py
Master orchestrator for the Closed-Loop ML-RL-XAI Empirical Replication Package.
Generates all datasets, executes D1-D5 across 30 runs, exports LaTeX tables, and renders figures.
"""

import os
import json
import pandas as pd
from src.corpus_generator import generate_all_corpora
from src.experiments_d1_d5 import (
    run_d1_predictive_accuracy,
    run_d2_and_d4_rl_ablation,
    run_d3_xai_fidelity,
    run_d5_scalability,
)
from src.stats_d10 import export_d1_latex_table, export_d3_human_study_table
from src.plot_results import generate_all_result_figures


def main() -> None:
    os.makedirs("results/raw_runs", exist_ok=True)
    os.makedirs("results/tables", exist_ok=True)
    os.makedirs("results/figures", exist_ok=True)

    print("=== STEP 1: Generating Multi-Tier Corpora & Traceability Worksheets ===")
    generate_all_corpora("data")

    df_t1 = pd.read_csv("data/tier1_featureide.csv")
    with open("data/tier1_constraints.json") as f:
        c_t1 = json.load(f)["clauses"]

    df_t2 = pd.read_csv("data/tier2_linux_kconfig.csv")
    with open("data/tier2_constraints.json") as f:
        c_t2 = json.load(f)["clauses"]

    df_user = pd.read_csv("data/practitioner_study_worksheet.csv")

    print("=== STEP 2: Running Dimension 1 (Predictive Accuracy - 30 Runs) ===")
    d1_t1 = run_d1_predictive_accuracy(df_t1, c_t1, "Tier1_FeatureIDE", n_runs=30)
    d1_t2 = run_d1_predictive_accuracy(df_t2, c_t2, "Tier2_LinuxKernel", n_runs=30)

    print("=== STEP 3: Running Dimension 2 (RL Optimization) & Dimension 4 (Ablation) ===")
    run_d2_and_d4_rl_ablation(df_t1, c_t1, "Tier1_FeatureIDE", n_runs=30)
    run_d2_and_d4_rl_ablation(df_t2, c_t2, "Tier2_LinuxKernel", n_runs=30)

    print("=== STEP 4: Running Dimension 3 (XAI Fidelity & Stability) ===")
    run_d3_xai_fidelity(df_t1, c_t1, "Tier1_FeatureIDE", n_runs=30)
    run_d3_xai_fidelity(df_t2, c_t2, "Tier2_LinuxKernel", n_runs=30)

    print("=== STEP 5: Running Dimension 5 (Sundermann 50-Model Scalability) ===")
    run_d5_scalability("data")

    print("=== STEP 6: Exporting Statistical LaTeX Tables (D10) ===")
    export_d1_latex_table(d1_t1, "results/tables/tab_res_d1_tier1.tex")
    export_d1_latex_table(d1_t2, "results/tables/tab_res_d1_tier2.tex")
    export_d3_human_study_table(df_user, "results/tables/tab_res_d3_practitioner.tex")

    print("=== STEP 7: Rendering Publication-Grade Section V Figures ===")
    generate_all_result_figures()
    print("=== PIPELINE COMPLETE: All traceability artifacts generated in data/ and results/ ===")


if __name__ == "__main__":
    main()

=== STEP 1: Generating Multi-Tier Corpora & Traceability Worksheets ===
Generated all benchmark tiers and data/corpus_manifest.csv
=== STEP 2: Running Dimension 1 (Predictive Accuracy - 30 Runs) ===


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py

=== STEP 3: Running Dimension 2 (RL Optimization) & Dimension 4 (Ablation) ===
=== STEP 4: Running Dimension 3 (XAI Fidelity & Stability) ===
=== STEP 5: Running Dimension 5 (Sundermann 50-Model Scalability) ===
=== STEP 6: Exporting Statistical LaTeX Tables (D10) ===
=== STEP 7: Rendering Publication-Grade Section V Figures ===
Generated all Section V figures in results/figures/
=== PIPELINE COMPLETE: All traceability artifacts generated in data/ and results/ ===
